# Phase 14.5 — smoke run

**This notebook contains no logic.** Every cell installs, invokes, or downloads.
The training code lives in the repo and is pinned to a commit, so what ran is
always recoverable from `git_commit` — the same provenance discipline
`evaluation/report.py` applies to scores.

**What this run is for.** Not accuracy. It proves the chain end to end:
targets decode, `torchlibrosa` works on Kaggle's torch/numpy, a checkpoint
round-trips Kaggle → local, resume works, and the saved file clears the
160MB floor that `PianoTranscription` silently enforces. **A terrible score
is the expected outcome of 500 steps and says nothing about the approach.**

## Before running
1. **Settings → Accelerator → GPU** (P100 or T4).
2. **Settings → Internet → On** (needed for pip and the checkpoint download).
3. **Add Data →** search `maestro-v3.0.0` and attach the public dataset.
   Then check the mount path in cell 2 — Kaggle names vary by uploader.
4. Set `COMMIT` below to the commit you want to run.

In [ ]:
# NOTE: the shell magics here build their command with an EXPLICIT
# f-string rather than `!cmd {VAR}`. That interpolation did not fire on
# Kaggle (measured 2026-08-18): the clone ran with a literal `{REPO}` and
# failed with "repository '{REPO}' does not exist".
COMMIT = "phase-14-training"   # a branch, tag, or full SHA
REPO = "https://github.com/ImSe4n/PTify.git"

# --no-deps is load-bearing: Kaggle's preinstalled torch is much newer than
# this project's local pin (measured: torch 2.10 / numpy 2.0 there vs
# 2.2/1.26 here). That is FINE — a checkpoint is a plain state_dict and
# crosses versions — but letting a resolver loose would reinstall torch and
# waste most of the session.
#
# The cost of --no-deps is naming every transitive import ourselves.
# `piano_transcription_inference` imports mido at module load, which the
# first Kaggle run discovered as a ModuleNotFoundError several minutes in.
!{f"pip install -q --no-deps git+{REPO}@{COMMIT}"}
!pip install -q --no-deps piano_transcription_inference torchlibrosa \
    mido pretty_midi librosa soundfile resampy audioread soxr lazy_loader msgpack

import importlib
missing = []
for m in ["mido", "pretty_midi", "librosa", "soundfile", "resampy", "soxr",
          "torchlibrosa", "piano_transcription_inference"]:
    try:
        importlib.import_module(m)
    except Exception as e:
        missing.append(f"{m}: {type(e).__name__} {e}")
print("MISSING:", missing or "none — all imports OK")

import torch, numpy
print("torch", torch.__version__, "| numpy", numpy.__version__)
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")


## 1. Clone the repo and find the MAESTRO mount

The segment index stores **relative** paths (`2011/MIDI-Unprocessed_....wav`),
so it only needs the directory those resolve against. If no dataset is listed,
it is not attached; if the year folders are one level deeper, adjust
`AUDIO_ROOT` accordingly.

In [ ]:
# NOTE: the shell magics here build their command with an EXPLICIT
# f-string rather than `!cmd {VAR}`. That interpolation did not fire on
# Kaggle (measured 2026-08-18): the clone ran with a literal `{REPO}` and
# failed with "repository '{REPO}' does not exist".
# Clone first: pip installs the PACKAGES but not the repo's data files, and
# the segment index in benchmarks/ is a data file.
!{f"git clone -q --depth 1 --branch {COMMIT} {REPO} /kaggle/working/PTify"}
!ls -la /kaggle/working/PTify/benchmarks/*.json

import glob, os
print("\n--- attached datasets ---")
for path in sorted(glob.glob("/kaggle/input/*/")):
    print(path)
    for sub in sorted(glob.glob(path + "*"))[:6]:
        print("   ", os.path.basename(sub))


In [ ]:
# Verified working on 2026-08-13 with the `alonhaviv/the-maestro-dataset-v3-0-0`
# public dataset. Kaggle nests an attached dataset under the uploader's name,
# so this path is specific to that upload — if you attach a different mirror,
# take the path from the listing above. What matters is that the directory
# CONTAINS the year folders (2004/ ... 2018/), because the index stores
# relative paths like `2011/MIDI-Unprocessed_....wav`.
AUDIO_ROOT = "/kaggle/input/datasets/alonhaviv/the-maestro-dataset-v3-0-0/maestro-v3.0.0"

import json, pathlib
index = json.load(open("/kaggle/working/PTify/benchmarks/maestro_segments_smoke.json"))
sample = index["tracks"][0]["audio_filename"]
probe = pathlib.Path(AUDIO_ROOT) / sample
print("probe:", probe)
print("EXISTS:", probe.exists(), "<- must be True before going further")

# The MIDI must be beside the audio; MAESTRO ships them together.
print("MIDI :", (pathlib.Path(AUDIO_ROOT) / index["tracks"][0]["midi_filename"]).exists())

## 2. Train

`--resume auto` starts fresh when there is no checkpoint and continues when
there is, so **re-running this cell after a session dies is the recovery
procedure** — there is no separate resume path to get wrong.

**`--batch-size 2 --accum-steps 4` — effective batch 8, but a quarter of the
memory.** The first GPU run OOMed on a T4 (14.56 GiB) at batch 8. This model
is far more memory-hungry than its 20M parameters suggest: it runs **four
parallel CRNN branches** (frame, onset, offset, velocity), each holding
activations over 1001 frames x 229 mel bins for the backward pass.
Accumulation keeps the effective batch identical — the gradient is provably
the same, see `test_gradient_accumulation_matches_a_full_batch`.

**`--no-amp` is deliberate.** With AMP on, the near-OOM state produced NaN in
all four heads at step 0 rather than an honest allocation failure; fp32 fails
loudly instead. Re-enable it only once a run is known good, and treat the
speedup as a Phase 15 optimisation rather than a requirement.

If it still OOMs, drop to `--batch-size 1 --accum-steps 8`. The error message
now tells you the exact flags to use.

`--save-every-seconds 660` (11 minutes here, 11 *hours* for a real run) is the
wall-clock trigger. Kaggle kills at a fixed hour regardless of step count, so
a step-only trigger on a slow dataloader can miss the deadline entirely.

In [ ]:
!cd /kaggle/working/PTify && python -m training.train \
    --index benchmarks/maestro_segments_smoke.json \
    --audio-root {AUDIO_ROOT} \
    --out /kaggle/working/checkpoints \
    --device cuda \
    --no-amp \
    --steps 500 \
    --batch-size 2 \
    --accum-steps 4 \
    --workers 2 \
    --log-every 25 \
    --validate-every 250 \
    --save-every-steps 250 \
    --save-every-seconds 660 \
    --resume auto

## 2. Train

`--resume auto` starts fresh when there is no checkpoint and continues when
there is, so **re-running this cell after a session dies is the recovery
procedure** — there is no separate resume path to get wrong.

`--save-every-seconds 660` (11 minutes here, 11 *hours* for a real run) is the
wall-clock trigger. Kaggle kills at a fixed hour regardless of step count, so
a step-only trigger on a slow dataloader can miss the deadline entirely.

Expect roughly **0.3s per step** on a P100 at batch 8. If it is far slower,
check `steps_per_s` in the log below — under ~15 segments/sec/worker means
the dataloader is starving the GPU, not that the GPU is slow.

In [ ]:
import json
log = [json.loads(l) for l in open("/kaggle/working/checkpoints/train_log.jsonl")]
train_rows = [r for r in log if "total" in r]
steps = [r["step"] for r in train_rows]

print("logged steps:", steps[:5], "...", steps[-5:])
print("first loss:", train_rows[0]["total"], "| last loss:", train_rows[-1]["total"])
print("steps/s:", [r["steps_per_s"] for r in train_rows][-3:])

# After a resume the step counter CONTINUES rather than restarting at 1.
# If it restarted, --resume auto did not find the checkpoint.
print("continued past a restart:", steps[-1] > len(steps))

## 3. The kill/resume drill

**Interrupt the training cell manually partway through**, then run it again.
The log must show `Resumed from step_N.pt` and the step counter must continue
rather than restart. This is the single most important thing this notebook
verifies — a 12-hour cap makes resume the only way a real run ever finishes.

In [ ]:
import sys; sys.path.insert(0, "/kaggle/working/PTify")
from training.model import assert_deployable

ckpt = "/kaggle/working/checkpoints/ptify-note-pedal.pth"
assert_deployable(ckpt)
import os; print("OK: %.1f MB" % (os.path.getsize(ckpt) / 1e6))

## 6. Download it

Kaggle output persistence has bitten people, so get the artifact off-box
before the session ends. For a real run (Phase 15+), push each save to a
private HF repo instead — losing 11 GPU-hours to a lost output is the failure
this guards against.

Then, locally:

```bash
python -c "from training.model import assert_deployable; assert_deployable('ptify-note-pedal.pth')"
```

and load it through `PianoTranscription(checkpoint_path=...)` to confirm it
produces notes. That local load-back is the actual Phase 14.5 gate.

In [ ]:
from IPython.display import FileLink
FileLink("/kaggle/working/checkpoints/ptify-note-pedal.pth")